<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.2-designing-reliability/notebooks/exercises-10.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.2 — Designing Reliability
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

Retry math, 3-state circuit breaker, multi-provider fallback, 4-level graceful degradation, SLI/SLO/SLA, burn-rate alerts, chaos testing.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.2: Designing Reliability')


## Ex 1: Retry Math — Why Retry + Jitter Works


In [ ]:
print('Retry math (failures multiply if independent):')
p = 0.02  # per-call transient failure rate
for attempts in [1, 2, 3, 5]:
    fail = p ** attempts
    print(f'  {attempts} attempts @ {p:.1%} each -> overall fail {fail*100:.4f}% (success {(1-fail)*100:.4f}%)')
print()
print('Retry only helps when failures are TRANSIENT and INDEPENDENT.')
print('Not transient: 400/401/403.  Correlated: provider-wide outage.')


## Ex 2: Retry Strategies Compared


In [ ]:
print('Retry Strategies:')
for strategy, when, risk in [
    ('No retry','Idempotency unknown, mutating writes','User sees every transient blip'),
    ('Fixed delay','Rare, predictable failures','Retry storm under correlated outage'),
    ('Exponential backoff','Transient 5xx/429 from APIs','Tail latency grows fast w/o cap'),
    ('Exp backoff + jitter','Production default','None - this is the answer'),
]:
    print(f'  {strategy:24s}: {when:40s} | Risk: {risk}')


## Ex 3: 3-State Circuit Breaker


In [ ]:
print('Circuit Breaker States:')
for state, behavior, exit_to in [
    ('CLOSED','Normal: all calls pass through','OPEN after 5 fails / 60s'),
    ('OPEN','Reject instantly, no upstream call','HALF_OPEN after 30s cooldown'),
    ('HALF_OPEN','One probe call allowed','CLOSED on success, OPEN on fail'),
]:
    print(f'  {state:10s}: {behavior:40s} | Exit: {exit_to}')
print()
print('Why 3 states (not 2)?')
print('HALF_OPEN prevents a thundering herd at the end of cooldown.')
print('Single probe = low risk way to test recovery.')


## Ex 4: Multi-Provider Fallback Uptime Math


In [ ]:
print('Combined uptime with independent providers:')
per_provider_uptime = 0.999
p_fail = 1 - per_provider_uptime
for n in [1, 2, 3, 4]:
    combined = 1 - (p_fail ** n)
    nines = -(-(combined - 1)).__str__()
    print(f'  {n} provider(s): {combined*100:.4f}% uptime')
print()
print('3 providers @ 99.9% each = 99.9999% (six nines) IF failures independent.')
print('Add per-provider circuit breakers so one bad provider is quarantined.')


## Ex 5: 4-Level Graceful Degradation


In [ ]:
print('Graceful Degradation Levels:')
for level, what, when in [
    ('FULL','Primary model + RAG + tools','Happy path'),
    ('REDUCED','Cheaper model, no tools','p99 latency > 1.5s'),
    ('CACHED','Last known good from cache','p99 > 3s or partial outage'),
    ('MINIMAL','Static branded message','Breaker open, all providers failing'),
]:
    print(f'  {level:10s}: {what:32s} | When: {when}')
print()
print('Principle: never return a 500 to the user. SOMETHING is always better than nothing.')


## Ex 6: SLI / SLO / SLA + Error Budget


In [ ]:
print('SLI / SLO / SLA:')
for term, definition, example in [
    ('SLI','The measurement','p99 latency, success ratio'),
    ('SLO','Internal target (stricter than SLA)','99.5% / 28d, p99 < 2s'),
    ('SLA','External contract w/ credits','99.0% / month or refund'),
    ('Error Budget','1 - SLO; spend on risk','0.5% downtime / 28d allowed'),
]:
    print(f'  {term:14s}: {definition:36s} | Ex: {example}')
print()
print('28-day budget math: SLO 99.5% -> 0.5% budget -> 201 minutes of downtime allowed.')


## Ex 7: Burn-Rate Alerts (Google SRE)


In [ ]:
print('Multi-window multi-burn-rate:')
for window, burn, action, meaning in [
    ('1h','14.4x','PAGE','Budget gone in 1h at this rate'),
    ('6h','6x','TICKET','Budget gone this week if sustained'),
    ('1d','3x','Slack alert','Trend warning'),
    ('3d','1x','Dashboard','Baseline drift, capacity check'),
]:
    print(f'  {window:4s} @ {burn:6s} -> {action:12s} | {meaning}')
print()
print('Alert on BURN RATE, not raw error rate -- budget is what the business cares about.')


---
## Recap
Retry (exp backoff + jitter) + 3-state circuit breaker + per-provider fallback + 4-level graceful degradation = six nines.
Retry math: 3 attempts at 2% fail -> 0.0008% overall fail (if transient + independent).
SLO stricter than SLA. Alert on burn rate (14.4x/1h page, 6x/6h ticket), not raw rate.
Chaos-test quarterly with toxiproxy / Litmus or your resilience is fiction.
